## 04a: Main Analyses

Fits the 28 primary tests. In the code, `HC` is the non-clinical control group (NCC).

**Normative deviation.** For each task, a regression fitted on HC (age as a natural cubic
spline, sex, education and device type, touchscreen or not) gives each MDD and BD participant a
deviation score

`D = (Y_observed − Ŷ_HC) / σ_HC`

where `σ_HC` is the residual standard deviation of the HC regression. Group-mean deviations
and the BD − MDD contrast are estimated with a no-intercept regression, `D ~ 0 + C(group)`,
with HC3 robust standard errors. As `D` is in HC standard-deviation units, each coefficient is
a Glass's Δ.

**Rest-activity phenotypes (MDD only).** Each phenotype (steps, WASO, TST, STV) is residualised
on age, sex, BMI, CCI and photoperiod and standardised to unit variance (`P_z`); `D` is then
regressed on it, `D ~ P_z` (HC3). Effect size: Pearson r. BD is not analysed, for lack of
power (see `04c`).

**Multiple testing.** 3 normative-deviation tests and 4 phenotype tests per task, × 4 tasks =
28 tests, corrected jointly with Benjamini-Hochberg (FDR 5%). Cells with fewer than 20
participants are suppressed and left out of the correction.

Model and sample definitions are in `src/analysis.py`. Output: `results/model_results.parquet`
(`04c` adds the MDES columns).

## Task 0: Setup

In [ ]:
import sys
import os
import pandas as pd
from statsmodels.stats.multitest import multipletests

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *
from src.analysis import (
    TASK_CONFIG, PHENOTYPES, MIN_CELL, MAIN_POOL_SIZE, RESULT_COLUMNS,
    prepare_cohort, qc_pass_mask, fit_normative_model, normative_sample,
    fitbit_sample, phenotype_complete, fit_group_model, fit_phenotype_association,
    result_row, suppressed_row,
)

OVERWRITE = True
RES_OUT = "nb04a"

# ── Load data ─────────────────────────────────────────────────────
cohort = prepare_cohort(load_parquet('cohort/cohort_master.parquet'))
wf     = load_parquet('features/wearable_features.parquet')

print(f'cohort          : {len(cohort):,} rows '
      f'(MDD {(cohort["group"]=="MDD").sum():,} / '
      f'BD {(cohort["group"]=="BD").sum():,} / '
      f'HC {(cohort["group"]=="HC").sum():,})')
print(f'wearable_features: {len(wf):,} rows '
      f'(MDD {(wf["group"]=="MDD").sum():,} / '
      f'BD {(wf["group"]=="BD").sum():,} / '
      f'HC {(wf["group"]=="HC").sum():,} across {wf["task"].nunique()} tasks)')
print(f'OVERWRITE       : {OVERWRITE}')

# One row per primary test (MAIN_POOL_SIZE = 28).
all_results = []

def _print_result(label, row):
    if row['suppressed']:
        print(f'  {label:<32} SUPPRESSED (n < {MIN_CELL})')
        return
    star = ' *' if row['p_value'] < 0.05 else ''
    r2_s = f'  R²={row["r2"]:.4f}' if row['r2'] is not None else ''
    if row['group'] == 'contrast':
        n_s = f'MDD={row["n_mdd"]:,}/BD={row["n_bd"]:,}'
    elif row['group'] == 'MDD':
        n_s = f'N={row["n_mdd"]:>5,}'
    elif row['group'] == 'HC':
        n_s = f'N={row["n_hc"]:>5,}'
    else:
        n_s = f'N={row["n_bd"]:>5,}'
    print(f'  {label:<32} {n_s}  '
          f'β={row["beta"]:+.3f} [{row["ci_lo"]:+.3f},{row["ci_hi"]:+.3f}]  '
          f'ES={row["effect_size"]:+.3f}{r2_s}  p={row["p_value"]:.4g}{star}')

def run_task(task, cohort, wf, results_rows):
    out_col = TASK_CONFIG[task]['outcome']

    print(f'\n{"─" * 64}')
    print(f'  {task.upper()}  ·  outcome: {out_col}')
    print(f'{"─" * 64}')

    qc_pass = qc_pass_mask(cohort, task)
    print(f'QC pass  MDD={int((qc_pass & (cohort["mdd_eligible_"+task].fillna(0)==1)).sum()):,}  '
          f'BD={int((qc_pass & (cohort["bd_eligible_"+task].fillna(0)==1)).sum()):,}  '
          f'HC={int((qc_pass & (cohort["group"]=="HC")).sum()):,}')

    # Normative model, fitted on HC
    m_norm, sigma_hc, hc = fit_normative_model(cohort, task)
    n_hc = len(hc)
    print(f'HC model   N={n_hc:,}  R²={m_norm.rsquared:.3f}  σ_HC={sigma_hc:.4f}')

    ab_mdd = normative_sample(cohort, task, 'MDD', m_norm, sigma_hc)
    ab_bd  = normative_sample(cohort, task, 'BD',  m_norm, sigma_hc)
    print(f'Sample Normative  MDD N={len(ab_mdd):,} (mean_D={ab_mdd["D"].mean():+.3f})  '
          f'BD N={len(ab_bd):,} (mean_D={ab_bd["D"].mean():+.3f})')

    # Normative deviation: D ~ 0 + C(group); each β is a Glass's Δ
    m_a, ct_a = fit_group_model(ab_mdd, ab_bd)

    for grp, n_mdd, n_bd in [('MDD', len(ab_mdd), None), ('BD', None, len(ab_bd))]:
        param = f'C(group)[{grp}]'
        results_rows.append(result_row(
            task=task, analysis='Normative', phenotype=None, group=grp,
            n_hc=n_hc, n_mdd=n_mdd, n_bd=n_bd,
            beta=m_a.params[param], se=m_a.bse[param],
            t=m_a.tvalues[param],   p=m_a.pvalues[param],
            effect_size=m_a.params[param], r2=None,
        ))
        _print_result(f'Normative  {grp} intercept', results_rows[-1])

    # BD − MDD contrast
    results_rows.append(result_row(
        task=task, analysis='Normative', phenotype=None, group='contrast',
        n_hc=n_hc, n_mdd=len(ab_mdd), n_bd=len(ab_bd),
        beta=float(ct_a.effect), se=float(ct_a.sd),
        t=float(ct_a.tvalue),    p=float(ct_a.pvalue),
        effect_size=float(ct_a.effect), r2=None,
    ))
    _print_result('Normative  BD − MDD', results_rows[-1])

    c_mdd = fitbit_sample(ab_mdd, task, wf, 'MDD')
    print(f'Sample Fitbit (MDD)  N={len(c_mdd):,}  wearable-eligible')

    # Rest-activity phenotypes: D ~ P_z per phenotype, MDD only.
    for pheno in PHENOTYPES:
        c_p     = phenotype_complete(c_mdd, pheno)
        n_mdd_c = len(c_p)

        if n_mdd_c < MIN_CELL:
            print(f'  Fitbit/{pheno:<18} MDD N={n_mdd_c} < {MIN_CELL}, suppressing')
            results_rows.append(suppressed_row(
                task=task, analysis='Fitbit', phenotype=pheno,
                group='MDD', n_hc=n_hc, n_mdd=n_mdd_c, n_bd=None,
            ))
            continue

        m_c, r_mdd, _ = fit_phenotype_association(c_p, task, pheno)
        results_rows.append(result_row(
            task=task, analysis='Fitbit', phenotype=pheno,
            group='MDD', n_hc=n_hc, n_mdd=n_mdd_c, n_bd=None,
            beta=m_c.params['P_z'], se=m_c.bse['P_z'],
            t=m_c.tvalues['P_z'],   p=m_c.pvalues['P_z'],
            effect_size=r_mdd, r2=r_mdd**2,
        ))
        _print_result(f'Fitbit/{pheno} MDD', results_rows[-1])

    return results_rows


## Task 1: GradCPT

Outcome: d′ (`gradcpt_dprime`). Higher is better, so impairment gives D < 0.

In [ ]:
all_results = run_task('gradcpt', cohort, wf, all_results)


## Task 2: Flanker

Outcome: rate-correct score interference cost (`flanker_rcs_interference`). Higher is worse,
so impairment gives D > 0.

In [ ]:
all_results = run_task('flanker', cohort, wf, all_results)


## Task 3: Delay Discounting

Outcome: ln k (`delaydiscounting_lnk`). Higher is more impulsive, so impairment gives D > 0.

In [ ]:
all_results = run_task('delaydiscounting', cohort, wf, all_results)


## Task 4: Emotion Recognition

Outcome: mean accuracy over happy, sad, fearful and angry faces (`emorecog_avg_accuracy`).
Higher is better, so impairment gives D < 0.

In [ ]:
all_results = run_task('emorecog', cohort, wf, all_results)


## Task 5: Benjamini-Hochberg Correction and Results Table

The non-suppressed p-values of the 28 tests are corrected jointly (FDR 5%). Effect sizes are
Glass's Δ for the normative-deviation rows (Δ_BD − Δ_MDD for the contrast) and Pearson r for
the phenotype rows.

In [ ]:
assert len(all_results) == MAIN_POOL_SIZE, (
    f'Expected {MAIN_POOL_SIZE} results (7 per task x 4 tasks: Normative x12 [MDD/BD/contrast] + '
    f'Fitbit MDD x16 [mean_steps/mean_waso/STV/mean_sleep_duration]); got {len(all_results)}. '
    'Sub-20 cells must emit a suppressed placeholder row, not be omitted.'
)

results_df = pd.DataFrame(all_results)

# Benjamini-Hochberg over the non-suppressed tests
non_supp = ~results_df['suppressed'].fillna(False)
_, p_bh, _, _ = multipletests(results_df.loc[non_supp, 'p_value'].values, method='fdr_bh')
results_df.loc[non_supp, 'p_bh']        = p_bh
results_df.loc[non_supp, 'significant'] = p_bh < 0.05

results_df = results_df[RESULT_COLUMNS].reset_index(drop=True)

print('\n' + '=' * 130)
print(f'  MAIN RESULTS SUMMARY  ({MAIN_POOL_SIZE}-test pool, BH-corrected, FDR = 5 %)')
print('=' * 130)
print(f'{"Task":<18} {"Analysis":>9} {"Phenotype":<14} {"Group":<10} '
      f'{"N_MDD":>7} {"N_BD":>7} {"N_HC":>7} '
      f'{"beta":>7} {"95% CI":>18} {"ES":>6} {"R2":>6} '
      f'{"p_raw":>8} {"p_BH":>8} {"sig":>4}')
print('-' * 130)

for _, row in results_df.iterrows():
    pheno = str(row['phenotype']) if pd.notna(row['phenotype']) else '—'
    if row['suppressed']:
        print(f'{row["task"]:<18} {row["analysis"]:>9} {pheno:<14} {row["group"]:<10} '
              f'{"SUPPRESSED":>52}')
        continue
    ci_str  = f'[{row["ci_lo"]:+.3f},{row["ci_hi"]:+.3f}]'
    sig     = '***' if row['significant'] else ''
    r2_str  = f'{row["r2"]:.4f}' if pd.notna(row['r2']) else '    —'
    n_mdd_s = f'{int(row["n_mdd"]):,}' if pd.notna(row['n_mdd']) else '      —'
    n_bd_s  = f'{int(row["n_bd"]):,}'  if pd.notna(row['n_bd'])  else '      —'
    n_hc_s  = f'{int(row["n_hc"]):,}'  if row['group'] == 'HC'  else '      —'
    print(
        f'{row["task"]:<18} {row["analysis"]:>9} {pheno:<14} {row["group"]:<10} '
        f'{n_mdd_s:>7} {n_bd_s:>7} {n_hc_s:>7} '
        f'{row["beta"]:>+7.3f} {ci_str:>18} '
        f'{row["effect_size"]:>+6.3f} {r2_str:>6} '
        f'{row["p_value"]:>8.4g} {row["p_bh"]:>8.4g} {sig:>4}'
    )

print('\n' + '=' * 130)
n_supp   = int(results_df['suppressed'].sum())
n_sig    = int(results_df['significant'].fillna(False).sum())
n_tested = int(non_supp.sum())
print(f'Significant after BH (FDR 5%), main {MAIN_POOL_SIZE}-test pool: {n_sig} / {n_tested} non-suppressed tests  '
      f'({n_supp} rows suppressed, n < 20)')

print('\n=== R² summary — rest-activity phenotypes, MDD (Pearson r²) ===')
r2_mdd = (
    results_df[
        (results_df['analysis'] == 'Fitbit') & (results_df['group'] == 'MDD')
        & ~results_df['suppressed']
    ]
    .pivot_table(index='phenotype', columns='task', values='r2')
    .round(4)
)
print(r2_mdd.to_string())

# Derived output — always saved.
save_res(RES_OUT, 'main_results.csv',  OVERWRITE, results_df,  file_type='csv')
save_parquet(results_df, 'results/model_results.parquet')
